# **Change raster resolution**

---

This notebook coarsens a list of single-band categorical raster maps (e.g., a land-cover map for one year) to one or more target resolutions, using the pixel-aggregation method described in:

> Pontius Jr., R.G. and Connors, J., 2009. **Range of categorical associations for comparison of maps with mixed pixels.** *Photogrammetric Engineering & Remote Sensing*, 75(8), pp.963-969. <https://www.ingentaconnect.com/content/asprs/pers/2009/00000075/00000008/art00003>

The paper shows that once several pure pixels are merged into one coarser pixel, that coarser pixel is no longer pure — it carries a *fractional membership* to every class it now spans (their Figure 1). This notebook implements exactly that aggregation step, so a hard-classified map can be turned into a soft, mixed-pixel map at whichever resolution is needed, ready to feed into the range-of-association equations from the same paper.

Follow Section 2 to point the notebook at your data, then run the remaining cells in order.

### What the notebook computes

For every raster in your list, at every target resolution you request, the notebook:
1. Reads the raster's native pixel size and nodata value directly from its own file metadata.
2. Groups neighboring pixels into blocks (e.g., 2x2, 4x4) according to how much coarser the target resolution is.
3. Computes, for each block, the fraction of pixels belonging to each class, ignoring nodata pixels in that calculation.
4. Writes the result as a compressed, multiband GeoTIFF, one band per class.

### Requirements for the rasters themselves

- **Single band.** Only band 1 of each file is read.
- **Pixel value equals an integer class code**, not a continuous quantity. Example: 0=water, 1=forest, 2=built.
- **Native pixel size stored in the file's own geotransform.** Read automatically; see Section 2 if a file's metadata is unreliable.
- **Nodata (optional but recommended).** If declared in the file, it is excluded from the aggregation automatically and carried through to the output. If not declared, every pixel is treated as valid and the notebook warns you.
- **Target resolution must be an integer multiple of each raster's native pixel size.** The notebook raises a clear error otherwise, naming the offending raster.

### Generated outputs

Results are written to the `OUTPUT_BASE_DIR` folder you set in Section 2:

| Folder | Contents | Produced in |
|---|---|---|
| `res_{resolution}m/` | One multiband GeoTIFF per input raster, storing fractional class membership at that target resolution (one subfolder per entry in `TARGET_RESOLUTIONS`) | Section 3 |


## **1. Environment Setup**

In [ ]:
# @title 1.1 Install Dependencies
# @markdown Run this cell only if you are using Google Colab
get_ipython().system('pip -q install rasterio')

In [ ]:
# @title 1.2 Import Modules

# Standard Library Imports
import os

# Third-party Data & Geospatial Imports
import numpy as np
import rasterio

In [ ]:
# @title 1.3 Configure Google Drive
# @markdown Run this cell only if your data is stored on Google Drive

try:
    from google.colab import drive
except ModuleNotFoundError:
    print("Google Drive mount skipped: this notebook is running outside Google Colab.")
else:
    drive.mount("/content/drive")

## **2. Input your data**

Before running Section 2.1, gather the raster maps you want to resample.

### What you need

1. **A list of raster file paths**, `RASTER_PATHS`. At least one path is required; there is no upper limit. The rasters do not need to share the same class set or the same native resolution — each is processed independently.
2. **A list of target resolutions**, `TARGET_RESOLUTIONS`, in the same spatial units as your rasters' CRS (e.g., meters for a projected CRS). Each value must be an integer multiple of the corresponding raster's native pixel size.
3. **An output folder**, `OUTPUT_BASE_DIR`. A separate subfolder is created per resolution (e.g., `res_20m/`, `res_40m/`), so outputs from different resolutions never mix.

`NATIVE_PIXEL_SIZE_OVERRIDE` is optional and normally left as `None`: the pipeline reads each raster's native pixel size directly from its own CRS/geotransform. Only set an entry here if a specific raster's file metadata is missing or unreliable, since in that case the auto-detected value cannot be trusted.


In [ ]:
# @title 2.1 Configure Your App { display-mode: "form" }
# @markdown ### 📂 Path Settings
# @markdown List of input raster file paths (edit with your own data):
RASTER_PATHS = ["/content/drive/MyDrive/toyData/etapa2_scenes_v2/scene1.tif"] #@param {type:"raw"}
# @markdown Base output folder (a subfolder is created per target resolution):
OUTPUT_BASE_DIR = "/content/drive/MyDrive/toyData/changeResolution/output" #@param {type:"string"}

# @markdown ### 📐 Resolution Settings
# @markdown List of target output resolutions, in the same spatial units as the input CRS:
TARGET_RESOLUTIONS = [20.0, 40.0] #@param {type:"raw"}
# @markdown Optional {raster_path: native_pixel_size} override. Leave as None unless a raster's metadata lacks a valid CRS/geotransform:
NATIVE_PIXEL_SIZE_OVERRIDE = None #@param {type:"raw"}

# Build the working variables consumed by the pipeline cells.
raster_paths = list(RASTER_PATHS)
target_resolutions = list(TARGET_RESOLUTIONS)
output_base_dir = OUTPUT_BASE_DIR
native_pixel_size_override = NATIVE_PIXEL_SIZE_OVERRIDE

# Validate inputs immediately, before any processing starts.
if not raster_paths:
    raise ValueError("Please add at least one path to RASTER_PATHS.")

missing_files = [
    path for path in raster_paths
    if not os.path.isfile(path)
]

if missing_files:
    print("Configuration needs attention.")
    print("Missing raster files:")
    for path in missing_files:
        print(f"  - {path}")
else:
    print(
        f"Ready to process {len(raster_paths)} raster(s) at "
        f"{len(target_resolutions)} resolution(s)."
    )

In [ ]:
# @title 2.2 Prepare Resolution-Changing Pipeline
# @markdown This cell defines the pixel-aggregation pipeline described in
# @markdown Pontius & Connors (2009) -- see the notebook introduction for the
# @markdown full citation. No user changes are required.


# 2.2.1 Retrieve the native pixel size of a raster
def get_pixel_size(
    raster_path: str
) -> float:
    """
    Retrieves the native pixel size (spatial resolution) of a raster.

    Parameters
    ----------
    raster_path : str
        Path to the input raster file.

    Returns
    -------
    float
        Pixel size in the raster's native CRS units. Assumes square
        pixels, i.e., equal resolution in the x and y directions.

    Raises
    ------
    ValueError
        If the raster does not have square pixels.
    """
    with rasterio.open(raster_path) as src:
        pixel_size_x = src.transform.a
        pixel_size_y = abs(src.transform.e)

    if not np.isclose(pixel_size_x, pixel_size_y):
        raise ValueError(
            "Non-square pixels are not supported by this pipeline."
        )

    return pixel_size_x


# 2.2.2 Compute the integer aggregation factor for a target resolution
def compute_aggregation_factor(
    native_pixel_size: float,
    target_pixel_size: float
) -> int:
    """
    Computes the integer aggregation factor (grain size g, following the
    notation of Pontius & Connors, 2009) required to coarsen a raster
    from its native resolution to a target resolution.

    Parameters
    ----------
    native_pixel_size : float
        Pixel size of the input raster.
    target_pixel_size : float
        Desired pixel size of the output raster. Must be an integer
        multiple of native_pixel_size.

    Returns
    -------
    int
        Aggregation factor g, where
        g = target_pixel_size / native_pixel_size.

    Raises
    ------
    ValueError
        If target_pixel_size is not an integer multiple of
        native_pixel_size, or if it is smaller than native_pixel_size.
    """
    ratio = target_pixel_size / native_pixel_size
    factor = round(ratio)

    if not np.isclose(ratio, factor):
        raise ValueError(
            f"target_pixel_size ({target_pixel_size}) is not an integer "
            f"multiple of native_pixel_size ({native_pixel_size})."
        )

    if factor < 1:
        raise ValueError(
            "target_pixel_size must be greater than or equal to "
            "native_pixel_size. This pipeline only coarsens resolution, "
            "it does not sharpen it."
        )

    return int(factor)


# 2.2.3 Crop a raster so its dimensions are divisible by the factor
def crop_to_multiple(
    class_grid: np.ndarray,
    factor: int
) -> np.ndarray:
    """
    Crops a 2D array so that both dimensions are exact multiples of a
    given aggregation factor, discarding excess rows and columns from
    the bottom and right edges.

    Parameters
    ----------
    class_grid : np.ndarray
        2D array of integer class codes.
    factor : int
        Aggregation factor that the output dimensions must be divisible
        by.

    Returns
    -------
    np.ndarray
        Cropped 2D array with height and width divisible by factor.
    """
    height, width = class_grid.shape
    cropped_height = (height // factor) * factor
    cropped_width = (width // factor) * factor

    if cropped_height != height or cropped_width != width:
        print(
            f"[WARNING] Cropping raster from {(height, width)} to "
            f"{(cropped_height, cropped_width)} to fit factor={factor}."
        )

    return class_grid[:cropped_height, :cropped_width]


# 2.2.4 Aggregate a categorical raster into fractional class membership
def aggregate_categorical_to_membership(
    categorical_array: np.ndarray,
    classes: list,
    factor: int,
    input_nodata=None,
    output_nodata: float = -1.0
) -> np.ndarray:
    """
    Aggregates a single-band categorical raster into fractional class
    membership at a coarser grain, computed on demand following the
    pixel-aggregation logic of Pontius & Connors (2009), Figure 1.

    Nodata pixels in the input (identified by input_nodata) are
    excluded from the calculation: each aggregated pixel's class
    fraction is computed over the valid pixels within its block only,
    not over the full block. Blocks with zero valid pixels are written
    as output_nodata in every band.

    Parameters
    ----------
    categorical_array : np.ndarray
        2D array of integer class codes, with dimensions divisible by
        factor.
    classes : list
        Ordered list of class codes to compute membership for. Should
        not include input_nodata.
    factor : int
        Aggregation factor (grain size g).
    input_nodata : int or float, optional
        Value in categorical_array that marks invalid pixels (e.g.,
        outside the study area, cloud-masked). If None (default), every
        pixel is treated as valid.
    output_nodata : float, optional
        Value written to every band of an aggregated pixel whose block
        contains zero valid input pixels (default -1.0, which falls
        outside the valid 0-1 membership range).

    Returns
    -------
    np.ndarray
        Array of shape (len(classes), height / factor, width / factor)
        with the fractional membership of each class per aggregated
        pixel, or output_nodata where no valid input pixels exist.
    """
    height, width = categorical_array.shape
    agg_height, agg_width = height // factor, width // factor

    if input_nodata is not None:
        valid_mask = categorical_array != input_nodata
    else:
        valid_mask = np.ones_like(categorical_array, dtype=bool)

    valid_counts = valid_mask.astype(np.float32).reshape(
        agg_height,
        factor,
        agg_width,
        factor
    ).sum(axis=(1, 3))
    has_valid_pixels = valid_counts > 0

    membership = np.full(
        (len(classes), agg_height, agg_width),
        output_nodata,
        dtype=np.float32
    )

    for class_idx, class_code in enumerate(classes):
        class_mask = (
            (categorical_array == class_code) & valid_mask
        ).astype(np.float32)
        class_counts = class_mask.reshape(
            agg_height,
            factor,
            agg_width,
            factor
        ).sum(axis=(1, 3))

        fraction = np.zeros_like(class_counts)
        fraction[has_valid_pixels] = (
            class_counts[has_valid_pixels] / valid_counts[has_valid_pixels]
        )
        membership[class_idx][has_valid_pixels] = fraction[has_valid_pixels]

    return membership


# 2.2.5 Build the affine transform for the aggregated raster
def build_aggregated_transform(
    original_transform: rasterio.Affine,
    factor: int
) -> rasterio.Affine:
    """
    Builds the affine transform for an aggregated raster, scaling pixel
    size by the aggregation factor while preserving the origin.

    Parameters
    ----------
    original_transform : rasterio.Affine
        Affine transform of the native-resolution raster.
    factor : int
        Aggregation factor applied to the raster.

    Returns
    -------
    rasterio.Affine
        Affine transform for the aggregated raster.
    """
    return rasterio.Affine(
        original_transform.a * factor,
        original_transform.b,
        original_transform.c,
        original_transform.d,
        original_transform.e * factor,
        original_transform.f
    )


# 2.2.6 Write a multiband fractional-membership raster, highly compressed
def write_membership_raster(
    membership_array: np.ndarray,
    class_names: list,
    output_path: str,
    transform: rasterio.Affine,
    crs,
    nodata=None
) -> str:
    """
    Writes a multiband GeoTIFF where each band stores the fractional
    membership of one class. Output is always written with DEFLATE
    compression at maximum level and a floating-point predictor, which
    is lossless and well suited to the smooth membership surfaces this
    pipeline produces.

    Parameters
    ----------
    membership_array : np.ndarray
        Array of shape (num_classes, height, width) with fractional
        membership values.
    class_names : list
        Names or codes for each class, in the same order as the first
        axis of membership_array. Used as band descriptions.
    output_path : str
        Full file path (including filename) for the output GeoTIFF.
    transform : rasterio.Affine
        Affine transform for the output raster.
    crs : rasterio.crs.CRS or str
        Coordinate Reference System of the output raster.
    nodata : float, optional
        Nodata value to declare in the output file, normally the same
        nodata value carried by the corresponding input raster. If
        None (default), no nodata value is declared, since the input
        raster had none either.

    Returns
    -------
    str
        The output file path.
    """
    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    num_classes, height, width = membership_array.shape

    meta = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": num_classes,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "compress": "deflate",
        "predictor": 3,
        "zlevel": 9,
        "tiled": True
    }
    if nodata is not None:
        meta["nodata"] = nodata

    with rasterio.open(output_path, "w", **meta) as dst:
        for band_idx in range(num_classes):
            dst.write(membership_array[band_idx], band_idx + 1)
            dst.set_band_description(
                band_idx + 1,
                str(class_names[band_idx])
            )

    uncompressed_bytes = membership_array.nbytes
    compressed_bytes = os.path.getsize(output_path)
    print(
        f"[LOG] Exported: {output_path} "
        f"(bands: {class_names}, shape: {height}x{width}, "
        f"compressed: {compressed_bytes} bytes vs "
        f"{uncompressed_bytes} bytes raw)"
    )

    return output_path


# 2.2.7 Orchestrate the full pipeline over a list of raster maps
def change_raster_list_resolution(
    raster_paths: list,
    target_pixel_size: float,
    output_dir: str,
    class_list: list = None,
    native_pixel_size_override: dict = None
) -> list:
    """
    Applies the pixel-aggregation logic of Pontius & Connors (2009) to a
    list of single-band categorical raster maps, converting each one
    from its native (pure-pixel) resolution to a coarser, user-specified
    resolution where pixels carry fractional class membership.

    Parameters
    ----------
    raster_paths : list of str
        Paths to the input single-band categorical rasters. Each raster
        is processed independently, so this function does not require
        the rasters to share the same class set or the same native
        resolution. Must contain at least one path.
    target_pixel_size : float
        Desired output pixel size, in the same spatial units as the
        input rasters' CRS. Must be an integer multiple of each
        raster's native pixel size.
    output_dir : str
        Directory where the output membership rasters will be saved.
    class_list : list, optional
        Explicit ordered list of class codes to use for every output
        raster. If None (default), the class list is derived
        independently for each raster from its unique pixel values.
    native_pixel_size_override : dict, optional
        Mapping of {raster_path: native_pixel_size} used to bypass
        automatic detection of a raster's native pixel size. By
        default (None), the native pixel size is read directly from
        each raster's own geotransform via get_pixel_size. Only needed
        for rasters whose file metadata does not carry a valid CRS or
        geotransform, since in that case the detected value cannot be
        trusted.

    Returns
    -------
    list of str
        Paths to the output multiband membership rasters, in the same
        order as raster_paths.

    Raises
    ------
    ValueError
        If raster_paths is empty.
    """
    if len(raster_paths) == 0:
        raise ValueError(
            "raster_paths must contain at least one raster file path."
        )

    output_paths = []

    for raster_path in raster_paths:
        with rasterio.open(raster_path) as src:
            class_grid = src.read(1)
            native_transform = src.transform
            crs = src.crs
            input_nodata = src.nodata

        if (
            native_pixel_size_override is not None
            and raster_path in native_pixel_size_override
        ):
            native_pixel_size = native_pixel_size_override[raster_path]
        else:
            native_pixel_size = get_pixel_size(raster_path)

        if input_nodata is None:
            print(
                f"[WARNING] {os.path.basename(raster_path)}: no nodata "
                "value found in the file metadata. All pixels will be "
                "treated as valid data."
            )

        print(
            f"[LOG] {os.path.basename(raster_path)}: detected native "
            f"pixel size = {native_pixel_size}, nodata = {input_nodata}"
        )

        factor = compute_aggregation_factor(
            native_pixel_size,
            target_pixel_size
        )

        class_grid = crop_to_multiple(class_grid, factor)
        classes = (
            class_list
            if class_list is not None
            else sorted(
                code for code in np.unique(class_grid).tolist()
                if code != input_nodata
            )
        )

        # The output keeps the same nodata value as the input raster,
        # so a pixel that was invalid before aggregation stays
        # identifiable as invalid after aggregation.
        output_nodata = input_nodata

        membership = aggregate_categorical_to_membership(
            class_grid,
            classes,
            factor,
            input_nodata=input_nodata,
            output_nodata=(
                output_nodata if output_nodata is not None else -1.0
            )
        )
        aggregated_transform = build_aggregated_transform(
            native_transform,
            factor
        )

        base_name = os.path.splitext(os.path.basename(raster_path))[0]
        output_path = os.path.join(
            output_dir,
            f"{base_name}.tif"
        )
        output_path = write_membership_raster(
            membership,
            classes,
            output_path,
            aggregated_transform,
            crs,
            nodata=output_nodata
        )
        output_paths.append(output_path)

    return output_paths

## **3. Apply the Pipeline**

In [ ]:
# @title 3.1 Run Pipeline for All Resolutions
output_paths_by_resolution = {}
for target_resolution in target_resolutions:
    resolution_dir = os.path.join(
        output_base_dir,
        f"res_{target_resolution:g}m"
    )
    output_paths_by_resolution[target_resolution] = change_raster_list_resolution(
        raster_paths=raster_paths,
        target_pixel_size=target_resolution,
        output_dir=resolution_dir,
        native_pixel_size_override=native_pixel_size_override
    )

print(
    f"\n[SUCCESS] Resolution change complete for {len(raster_paths)} "
    f"raster(s), at {len(target_resolutions)} resolution(s)!"
)

## **4. Verification**

Reads every output raster back and checks that fractional membership sums to 1.0 at every valid pixel -- the one property that must always hold, regardless of which rasters or resolutions were used as input.

In [ ]:
# @title 4.1 Check Membership Sums to 1
all_checks_passed = True
for target_resolution, output_paths in output_paths_by_resolution.items():
    for output_path in output_paths:
        with rasterio.open(output_path) as src:
            membership_array = src.read()
            membership_sum = membership_array.sum(axis=0)
            num_bands = src.count
            pixel_size = src.transform.a
            output_nodata = src.nodata

        if output_nodata is not None:
            valid_pixels_mask = ~np.any(
                np.isclose(membership_array, output_nodata),
                axis=0
            )
        else:
            valid_pixels_mask = np.ones(
                membership_sum.shape,
                dtype=bool
            )

        sums_to_one = np.allclose(
            membership_sum[valid_pixels_mask],
            1.0
        )
        all_checks_passed &= sums_to_one

        print(
            f"[CHECK] {os.path.basename(output_path)} "
            f"(target={target_resolution}, pixel_size={pixel_size}, "
            f"bands={num_bands}) -> sums to 1: {sums_to_one}"
        )

print(f"\n[RESULT] All checks passed: {all_checks_passed}")